# 01 — FASE 1: Ingestão

> **A pergunta desta fase:** como trago o dado de fora para dentro **sem perder
> informação**, **sem derrubar a fonte** e de um jeito que eu consiga
> **repetir amanhã**?

Ingestão parece a parte fácil ("é só um `requests.get`"). Não é. É onde moram os
problemas mais chatos de um pipeline: paginação, limite de requisições, falha de
rede, mudança silenciosa no formato, reprocessamento duplicando dado.

Neste notebook vamos construir a ingestão em camadas de entendimento:

1. uma requisição manual — para ver o problema aparecer;
2. paginação — porque a API não entrega tudo de uma vez;
3. controle de vazão e retry — porque a rede falha e a API tem limite;
4. a camada **raw** — o que gravar, como particionar e por que guardar hash;
5. **idempotência** — rodar duas vezes tem de dar o mesmo resultado.

In [2]:
# --- Preparação do ambiente (rode esta célula primeiro) --------------------
import sys
from pathlib import Path

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ / "src"))

import pandas as pd

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)

print("Raiz do projeto:", RAIZ)
print("Python:", sys.version.split()[0])


Raiz do projeto: c:\dev\Estudo-MDS
Python: 3.12.10


## 1. Uma requisição, na mão

Começamos do jeito mais simples possível para o problema aparecer sozinho.

In [3]:
import requests

from f1_pipeline import config

url = f"{config.API_BASE_URL}/2024/results.json"
resposta = requests.get(url, params={"limit": 100, "offset": 0}, timeout=30)
payload = resposta.json()

mrdata = payload["MRData"]
print("limit  :", mrdata["limit"], "  <- quantos vieram nesta resposta")
print("offset :", mrdata["offset"], "  <- de onde começou")
print("total  :", mrdata["total"], "  <- quantos existem no total")

corridas = mrdata["RaceTable"]["Races"]
print("\nCorridas neste payload:", len(corridas))
print("Resultados na primeira corrida:", len(corridas[0]["Results"]))


limit  : 100   <- quantos vieram nesta resposta
offset : 0   <- de onde começou
total  : 479   <- quantos existem no total

Corridas neste payload: 6
Resultados na primeira corrida: 20


### O problema já apareceu

`total` diz que existem **479** resultados na temporada de 2024, mas a resposta
trouxe **100**. Se você parasse aqui — e muita gente para — seu dashboard estaria
mostrando um quinto da realidade, sem nenhuma mensagem de erro.

> **Lição 1.** Uma ingestão que "funciona" não é uma ingestão que roda sem erro:
> é uma ingestão que traz **tudo** o que deveria trazer. Erro silencioso é o pior
> tipo de erro em dados.

## 2. Paginação

O contrato é o de sempre em APIs REST: peço `limit` registros a partir de um
`offset`, e a resposta me diz o `total`. Enquanto `offset + limit < total`,
existe mais página.

In [4]:
def paginar_manualmente(url: str, limite: int = 100):
    """Versão didática: percorre todas as páginas e devolve os payloads."""
    offset, paginas = 0, []
    while True:
        r = requests.get(url, params={"limit": limite, "offset": offset}, timeout=30)
        p = r.json()["MRData"]
        paginas.append(p)
        total = int(p["total"])
        print(f"  página {len(paginas)}: offset={offset:>3} de {total}")
        offset += limite
        if offset >= total:
            return paginas


paginas = paginar_manualmente(url)
resultados = sum(len(c["Results"]) for p in paginas for c in p["RaceTable"]["Races"])
print(f"\nTotal de resultados coletados: {resultados}")


  página 1: offset=  0 de 479
  página 2: offset=100 de 479
  página 3: offset=200 de 479
  página 4: offset=300 de 479
  página 5: offset=400 de 479

Total de resultados coletados: 479


## 3. Rate limit e retry: os dois cuidados que separam script de pipeline

A versão acima funciona... até não funcionar. Dois problemas reais:

**(a) Limite de requisições.** A API permite ~4 chamadas por segundo. Um `for`
sem pausa dispara dezenas por segundo, leva `429 Too Many Requests` e, no limite,
bloqueio de IP. A solução é **throttling**: garantir um intervalo mínimo entre
chamadas.

**(b) Falha transitória.** Rede cai, servidor devolve 503, o DNS engasga. Isso
não é exceção — é rotina. A solução é **retry com backoff exponencial**: tentar
de novo esperando 1s, 2s, 4s, 8s... Repetir imediatamente só piora a situação de
um servidor sobrecarregado.

O `ErgastClient` do projeto implementa os dois. Vamos olhar as partes que
importam.

In [5]:
import inspect

from f1_pipeline.ingestion import ErgastClient

print(inspect.getsource(ErgastClient._throttle))


    def _throttle(self) -> None:
        """Garante um intervalo mínimo entre duas requisições."""
        elapsed = time.monotonic() - self._last_request_at
        if elapsed < self.min_interval:
            time.sleep(self.min_interval - elapsed)
        self._last_request_at = time.monotonic()



In [6]:
# O retry é declarativo, com a biblioteca `tenacity`:
print(inspect.getsource(ErgastClient.fetch_page))


    def fetch_page(self, path: str, *, limit: int, offset: int) -> ApiPage:
        """Busca UMA página. O decorator de retry fica na função interna."""

        @retry(
            retry=retry_if_exception_type((ApiError, requests.RequestException)),
            stop=stop_after_attempt(self.max_retries),
            wait=wait_exponential(multiplier=1, min=1, max=30),
            reraise=True,
        )
        def _do_request() -> ApiPage:
            self._throttle()
            url = f"{self.base_url}/{path.strip('/')}.json"
            params = {"limit": limit, "offset": offset}
            started = time.monotonic()
            response = self.session.get(url, params=params, timeout=self.timeout)
            elapsed_ms = int((time.monotonic() - started) * 1000)
            self.request_count += 1

            if response.status_code == 429:
                retry_after = float(response.headers.get("Retry-After", 5))
                logger.warning(
                    "429 Too Many 

### Traduzindo o `@retry`

```python
@retry(
    retry=retry_if_exception_type((ApiError, requests.RequestException)),
    stop=stop_after_attempt(5),                       # desiste na 5ª tentativa
    wait=wait_exponential(multiplier=1, min=1, max=30),  # 1s, 2s, 4s, 8s...
    reraise=True,                                     # falhou de vez? propague
)
```

Duas decisões importantes aí dentro:

* **Só reenviamos o que vale a pena.** `429` e `5xx` são temporários — tentamos
  de novo. Já um `404` significa "esse recurso não existe": repetir mil vezes não
  vai fazer existir. Por isso o cliente separa esses casos.
* **`reraise=True`.** Se acabaram as tentativas, o erro sobe e o pipeline falha.
  Falhar alto é melhor do que gravar dado pela metade em silêncio.

In [7]:
# O cliente em ação: ele pagina sozinho e informa o que está fazendo.
with ErgastClient() as cliente:
    paginas_do_cliente = list(cliente.iter_pages("2024/driverstandings"))
    print("\nRequisições feitas:", cliente.request_count)

pagina = paginas_do_cliente[0]
print("URL final :", pagina.url)
print("Coletado  :", pagina.fetched_at)
print("Resposta  :", pagina.elapsed_ms, "ms")


22:01:14 | INFO    | f1_pipeline.ingestion.client | GET 2024/driverstandings | offset=0 | total=24 | 1092ms

Requisições feitas: 1
URL final : https://api.jolpi.ca/ergast/f1/2024/driverstandings.json?limit=100&offset=0
Coletado  : 2026-09-05T01:01:14+00:00
Resposta  : 1092 ms


## 4. A camada RAW

Agora a decisão mais importante da fase: **o que gravar?**

A tentação é gravar já bonitinho, em tabela, com os nomes que a gente gosta.
Resista. A camada raw guarda o payload **exatamente como veio**.

**Por que isso vale o espaço em disco:**

| Situação real | Com raw | Sem raw |
|---|---|---|
| Descobri um bug na transformação | reprocesso em segundos | preciso rebaixar tudo da API |
| A API saiu do ar / mudou o contrato | tenho o histórico | perdi o passado |
| Auditoria pergunta "de onde veio esse número?" | mostro o JSON e o hash | mostro um `SELECT` e uma promessa |
| Quero uma coluna que descartei na hora | está lá | perdida |

### Particionamento

Gravamos assim:

```
data/raw/results/season=2024/page_000.json
data/raw/results/season=2024/page_001.json
data/raw/results/season=2024/_manifest.json
```

O padrão `coluna=valor` no nome da pasta é **Hive partitioning** — lido
nativamente por Spark, DuckDB, Athena e pyarrow. Além de organizar, ele permite
**reprocessar só a partição que mudou** em vez de tudo.

In [8]:
from f1_pipeline.ingestion import get_endpoint, ingest_endpoint, partition_path

manifesto = ingest_endpoint(get_endpoint("races"), 2024)

print("\nPartição:", partition_path("races", 2024))
print("Arquivos gravados:", manifesto["arquivos_gravados"])
print("Registros informados pela API:", manifesto["registros_informados_pela_api"])


22:01:44 | INFO    | f1_pipeline.ingestion.client | GET 2024/races | offset=0 | total=24 | 983ms
22:01:44 | INFO    | f1_pipeline.ingestion.jobs   | RAW gravado: races                  season=2024 | 1 arquivo(s) | 24 registro(s) | 1.00s

Partição: C:\dev\Estudo-MDS\data\raw\races\season=2024
Arquivos gravados: 1
Registros informados pela API: 24


### O manifesto: a certidão de nascimento do dado

Junto de cada partição gravamos um `_manifest.json`. Ele responde perguntas que
sempre aparecem meses depois:

* **quando** este dado foi coletado?
* de **qual URL**, exatamente?
* quantas requisições, quantas páginas, quanto tempo?
* o arquivo foi **alterado** depois de gravado? (é para isso que serve o hash
  SHA-256)

Isso é **linhagem de dados** (*data lineage*) na sua forma mais simples e mais
honesta.

In [9]:
import json

print(json.dumps({k: v for k, v in manifesto.items() if k != "requisicoes"}, indent=2, ensure_ascii=False)[:1400])


{
  "endpoint": "races",
  "descricao": "Calendário da temporada: cada GP, sua data e seu circuito.",
  "temporada": 2024,
  "ingerido_em": "2026-09-05T01:01:44+00:00",
  "duracao_segundos": 1.0,
  "arquivos_gravados": 1,
  "requisicoes_http": 1,
  "blocos_no_json": 24,
  "registros_informados_pela_api": 24,
  "por_rodada": false,
  "base_url": "https://api.jolpi.ca/ergast/f1",
  "arquivos": [
    {
      "arquivo": "page_000.json",
      "bytes": 26683,
      "blocos_no_json": 24,
      "sha256": "96e33359d8da09993752990edf819e1ce77a7bf89570ba99afa497106ad10059"
    }
  ]
}


In [10]:
# O hash detecta qualquer alteração no arquivo bruto, mesmo de 1 byte.
from f1_pipeline.utils.io import file_digest

arquivo = partition_path("races", 2024) / "page_000.json"
print("sha256 registrado no manifesto:", manifesto["arquivos"][0]["sha256"][:32], "...")
print("sha256 recalculado agora      :", file_digest(arquivo)[:32], "...")
print("Íntegro?", file_digest(arquivo) == manifesto["arquivos"][0]["sha256"])


sha256 registrado no manifesto: 96e33359d8da09993752990edf819e1c ...
sha256 recalculado agora      : 96e33359d8da09993752990edf819e1c ...
Íntegro? True


## 5. Nem todo endpoint é igual: ingestão em *fan-out*

Um detalhe que só aparece quando você tenta: **`/2024/pitstops` devolve HTTP 400**.
A API só aceita paradas de box corrida a corrida: `/2024/1/pitstops`.

Isso é comuníssimo em APIs reais, e a solução é o padrão **fan-out**: uma
requisição por rodada. Note a dependência que isso cria — para pedir "as paradas
da rodada 5" eu preciso *antes* saber que existe uma rodada 5, ou seja, preciso
do calendário já ingerido.

No projeto isso está declarado no catálogo de endpoints, não escondido no meio do
código:

In [11]:
from f1_pipeline.ingestion import ENDPOINTS

pd.DataFrame(
    [
        {
            "endpoint": e.name,
            "caminho": e.path_template,
            "por_rodada": e.por_rodada,
            "o que traz": e.description,
        }
        for e in ENDPOINTS
    ]
)


,endpoint,caminho,por_rodada,o que traz
0,races,{season}/races,False,"Calendário da temporada: cada GP, sua data e s..."
1,results,{season}/results,False,Resultado final de cada piloto em cada GP (a t...
2,qualifying,{season}/qualifying,False,"Tempos de Q1, Q2 e Q3 — define o grid de largada."
3,pit_stops,{season}/{round}/pitstops,True,"Cada parada nos boxes: volta, horário e duração."
4,driver_standings,{season}/driverstandings,False,Classificação final do campeonato de pilotos.
5,constructor_standings,{season}/constructorstandings,False,Classificação final do campeonato de construto...


## 6. Idempotência: rodar duas vezes não pode estragar nada

**Idempotente** = executar N vezes produz o mesmo estado final que executar uma
vez. É o que permite reprocessar sem medo depois de uma falha no meio.

Nossa ingestão é idempotente porque cada partição é **sobrescrita por inteiro**,
nunca acrescentada. Não existe "acrescentar página duplicada".

E, quando o objetivo é só completar o que falta, `overwrite=False` pula as
partições que já existem — uma **carga incremental**.

In [12]:
from f1_pipeline.ingestion import ingest_all

# Como as partições já existem, esta chamada não gasta nenhuma requisição:
# ela apenas confirma o que já está em disco. Troque para overwrite=True
# quando quiser realmente rebaixar tudo da API.
resumo = ingest_all(config.SEASONS, overwrite=False)
resumo


22:02:44 | INFO    | f1_pipeline.ingestion.jobs   | Iniciando ingestão | temporadas=[2021, 2022, 2023, 2024] | endpoints=['races', 'results', 'qualifying', 'pit_stops', 'driver_standings', 'constructor_standings']
22:02:44 | INFO    | f1_pipeline.ingestion.jobs   | Partição races/season=2021 já existe - pulando (overwrite=False)
22:02:44 | INFO    | f1_pipeline.ingestion.jobs   | Partição results/season=2021 já existe - pulando (overwrite=False)
22:02:44 | INFO    | f1_pipeline.ingestion.jobs   | Partição qualifying/season=2021 já existe - pulando (overwrite=False)
22:02:44 | INFO    | f1_pipeline.ingestion.jobs   | Partição pit_stops/season=2021 já existe - pulando (overwrite=False)
22:02:44 | INFO    | f1_pipeline.ingestion.jobs   | Partição driver_standings/season=2021 já existe - pulando (overwrite=False)
22:02:44 | INFO    | f1_pipeline.ingestion.jobs   | Partição constructor_standings/season=2021 já existe - pulando (overwrite=False)
22:02:44 | INFO    | f1_pipeline.ingestion.job

,endpoint,temporada,requisicoes,arquivos,registros,bytes,duracao_s,ingerido_em
0,races,2021,1,1,22,21402,0.95,2026-09-04T22:17:32+00:00
1,results,2021,5,5,440,570628,3.03,2026-09-04T22:17:35+00:00
2,qualifying,2021,5,5,439,379647,4.94,2026-09-04T22:17:40+00:00
3,pit_stops,2021,22,22,798,177809,15.88,2026-09-04T22:17:56+00:00
4,driver_standings,2021,1,1,21,18473,0.73,2026-09-04T22:17:57+00:00
5,constructor_standings,2021,1,1,10,4542,0.67,2026-09-04T22:17:58+00:00
6,races,2022,1,1,22,24459,0.67,2026-09-04T22:17:58+00:00
7,results,2022,5,5,440,585081,2.75,2026-09-04T22:18:01+00:00
8,qualifying,2022,5,5,440,377155,4.83,2026-09-04T22:18:06+00:00
9,pit_stops,2022,22,22,805,179768,16.56,2026-09-04T22:18:22+00:00


## 7. Auditando o que foi ingerido

Como todo manifesto fica em disco, conseguimos montar um relatório da ingestão
a qualquer momento — sem tocar na API.

In [13]:
from f1_pipeline.ingestion import load_manifests

auditoria = load_manifests()
print(f"{len(auditoria)} partições ingeridas\n")

visao = (
    auditoria.groupby("endpoint", as_index=False)
    .agg(
        temporadas=("temporada", "nunique"),
        requisicoes=("requisicoes", "sum"),
        arquivos=("arquivos", "sum"),
        registros=("registros", "sum"),
        mb=("bytes", lambda s: round(s.sum() / 1024 / 1024, 2)),
        segundos=("duracao_s", "sum"),
    )
    .sort_values("registros", ascending=False)
)
visao


24 partições ingeridas



,endpoint,temporadas,requisicoes,arquivos,registros,mb,segundos
2,pit_stops,4,91,91,3340,0.71,119.70
5,results,4,20,20,1799,2.29,10.51
3,qualifying,4,20,20,1798,1.48,22.60
4,races,4,4,4,90,0.09,3.31
1,driver_standings,4,4,4,89,0.07,2.74
0,constructor_standings,4,4,4,40,0.02,2.52


In [14]:
from f1_pipeline.utils.io import describe_layer, human_size

inventario = describe_layer(config.RAW_DIR)
print("Arquivos na camada raw:", len(inventario))
print("Tamanho total         :", human_size(inventario["bytes"].sum()))
inventario.head(8)


Arquivos na camada raw: 167
Tamanho total         : 4.7 MB


,arquivo,tamanho,bytes,modificado_em
0,constructor_standings\season=2021\_manifest.json,922.0 B,922,2026-09-04 19:17
1,constructor_standings\season=2021\page_000.json,4.4 KB,4542,2026-09-04 19:17
2,constructor_standings\season=2022\_manifest.json,922.0 B,922,2026-09-04 19:18
3,constructor_standings\season=2022\page_000.json,4.4 KB,4535,2026-09-04 19:18
4,constructor_standings\season=2023\_manifest.json,922.0 B,922,2026-09-04 19:19
5,constructor_standings\season=2023\page_000.json,4.4 KB,4537,2026-09-04 19:19
6,constructor_standings\season=2024\_manifest.json,921.0 B,921,2026-09-04 19:20
7,constructor_standings\season=2024\page_000.json,4.4 KB,4508,2026-09-04 19:20


## 8. O que fica desta fase

* **Grave o cru.** A camada raw é a sua rede de segurança e a sua auditoria.
* **Pagine sempre.** `total` da resposta é o contrato; ignore-o e você perde dado
  em silêncio.
* **Seja educado com a fonte.** Throttling + `User-Agent` identificando quem é
  você. APIs públicas são bens comuns.
* **A rede falha; planeje isso.** Retry com backoff exponencial, e falhe alto
  quando acabarem as tentativas.
* **Particione.** `season=2024/` não é organização estética: é o que permite
  reprocessar um pedaço só.
* **Registre a linhagem.** Manifesto com URL, horário e hash responde perguntas
  que aparecem meses depois.

### Exercícios

1. **Sprints.** A partir de 2021 existem corridas sprint (`/{season}/sprint`).
   Elas valem pontos e **não estão** na nossa carga — por isso os totais deste
   projeto ficam abaixo do campeonato oficial. Acrescente o endpoint em
   `src/f1_pipeline/ingestion/endpoints.py` e refaça a ingestão.
2. **Voltas.** `/{season}/{round}/laps` traz o tempo de cada volta de cada
   piloto. Estime quantas requisições isso custaria para uma temporada
   (dica: ~1.100 registros por GP, 100 por página) e decida se vale a pena.
3. **Carga incremental de verdade.** Hoje `overwrite=False` pula a partição
   inteira. Como você faria para rebaixar apenas a última rodada de uma
   temporada em andamento?

---

**Próximo:** [`02_qualidade_great_expectations.ipynb`](02_qualidade_great_expectations.ipynb) —
o dado chegou, mas dá para confiar nele?